# Análisis exploratorio — datos scrapeados

Exploración del dataset de titulares uruguayos: volumen, fuentes, etiquetas GPT y distribución temporal.

**Fuente:** export CSV más reciente en `data/exports/`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "config.yml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import importlib
import mindful_news.training.data as _data
import mindful_news.training.preprocess as _preprocess

importlib.reload(_preprocess)
importlib.reload(_data)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 110

from mindful_news.training.data import latest_export_csv, load_labeled_headlines, make_temporal_splits

export_path = latest_export_csv()
print("Export:", export_path)
df = load_labeled_headlines(str(export_path) if export_path else None)
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df["scraped_at"] = pd.to_datetime(df["scraped_at"], errors="coerce")
df["titulo_len"] = df["titulo"].str.len()
df.head(3)


In [ ]:
summary = {
    "filas": len(df),
    "medios": df["medio"].nunique(),
    "temas": df["tema"].nunique(),
    "cargas": df["carga"].nunique(),
    "desde": df["split_date"].min(),
    "hasta": df["split_date"].max(),
}
pd.Series(summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

medio_counts = df["medio"].value_counts()
axes[0].barh(medio_counts.index, medio_counts.values, color=sns.color_palette()[0])
axes[0].set_title("Titulares por medio")
axes[0].set_xlabel("Cantidad")

tema_counts = df["tema"].value_counts()
sns.barplot(x=tema_counts.values, y=tema_counts.index, ax=axes[1], orient="h")
axes[1].set_title("Distribución de temas (GPT)")
axes[1].set_xlabel("Cantidad")
plt.tight_layout()
plt.show()


In [ ]:
order = ["baja", "media", "alta"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x="carga", order=order, ax=axes[0])
axes[0].set_title("Distribución de carga emocional")

ct = pd.crosstab(df["tema"], df["carga"]).loc[:, order]
sns.heatmap(ct, annot=True, fmt="d", cmap="Blues", ax=axes[1])
axes[1].set_title("Tema × carga")
plt.tight_layout()
plt.show()


In [ ]:
daily = df.set_index("split_date").resample("D").size()
weekly_medio = df.groupby([pd.Grouper(key="split_date", freq="W"), "medio"]).size().unstack(fill_value=0)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=False)
daily.plot(ax=axes[0], color="#4C72B0")
axes[0].set_title("Titulares por día")
axes[0].set_ylabel("Cantidad")

weekly_medio.plot(kind="area", stacked=True, ax=axes[1], alpha=0.85)
axes[1].set_title("Volumen semanal por medio")
axes[1].set_ylabel("Cantidad")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["titulo_len"], bins=30, kde=True, ax=axes[0])
med = df["titulo_len"].median()
axes[0].axvline(med, ls="--", c="red", label=f"mediana={med:.0f}")
axes[0].set_title("Longitud de titulares (chars)")
axes[0].legend()

tema_medio = pd.crosstab(df["medio"], df["tema"], normalize="index") * 100
sns.heatmap(tema_medio, cmap="YlOrRd", ax=axes[1])
axes[1].set_title("% tema por medio (filas normalizadas)")
plt.tight_layout()
plt.show()

print("Longitud min/med/max:", df["titulo_len"].min(), med, df["titulo_len"].max())


In [ ]:
splits = make_temporal_splits(df)
split_sizes = pd.Series({k: len(v) for k, v in splits.items()})
display(split_sizes)

print("Rango temporal por split:")
for name, part in splits.items():
    start = part["split_date"].min()
    end = part["split_date"].max()
    print(f"  {name:5s}: {start} -> {end} ({len(part)} filas)")


In [ ]:
for label_col in ["tema", "carga"]:
    print(f"\n=== Ejemplos por {label_col} ===")
    for label, group in df.groupby(label_col):
        sample = group.sample(min(2, len(group)), random_state=42)
        print(f"\n-- {label} --")
        for _, row in sample.iterrows():
            print(f"  [{row['medio']}] {row['titulo'][:100]}")
